# UPI Transaction Failure Prediction Model

This notebook trains machine learning models to predict the probability of UPI transaction failures using a synthetic UPI dataset.

Steps:
1. Load dataset
2. Feature engineering
3. Data preprocessing
4. Train/Test split
5. Model training
6. Model evaluation
7. Prediction demo

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

## Load Dataset

In [ ]:
df = pd.read_csv('../data/upi_transactions.csv')
df.head()

## Feature Engineering

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek

## Extract Bank Identifiers

In [ ]:
df['Sender_Bank'] = df['Sender UPI ID'].apply(lambda x: x.split('@')[1])
df['Receiver_Bank'] = df['Receiver UPI ID'].apply(lambda x: x.split('@')[1])

## Encode Categorical Data

In [ ]:
le = LabelEncoder()

df['Sender_Bank'] = le.fit_transform(df['Sender_Bank'])
df['Receiver_Bank'] = le.fit_transform(df['Receiver_Bank'])

df['Status'] = df['Status'].map({'SUCCESS':0,'FAILED':1})

## Feature Selection

In [ ]:
features = ['Amount (INR)','Hour','DayOfWeek','Sender_Bank','Receiver_Bank']

X = df[features]
y = df['Status']

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print('Training samples:', X_train.shape)
print('Testing samples:', X_test.shape)

## Random Forest Model

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

print('Random Forest Accuracy:', accuracy_score(y_test, rf_preds))

## XGBoost Model

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)

print('XGBoost Accuracy:', accuracy_score(y_test, xgb_preds))

## Model Evaluation

In [ ]:
print(classification_report(y_test, xgb_preds))

prob = xgb.predict_proba(X_test)[:,1]

print('ROC AUC Score:', roc_auc_score(y_test, prob))

## Prediction Demo

In [ ]:
sample_transaction = pd.DataFrame({
    'Amount (INR)': [2500],
    'Hour': [20],
    'DayOfWeek': [4],
    'Sender_Bank': [2],
    'Receiver_Bank': [1]
})

failure_prob = xgb.predict_proba(sample_transaction)[0][1]

print('Predicted Failure Probability:', round(failure_prob,2))

if failure_prob > 0.6:
    print('⚠ High chance of transaction failure')
else:
    print('✅ Transaction likely to succeed')